# 04 — Spatially-Aware Conformal Prediction (SACP)

**Phase 4 of FedSafe-Fuse.** Runs on Google Colab Free + T4.

## What this notebook does
1. Loads the FedSafe-Fuse + FIPCA checkpoint from Phase 3.
2. Runs inference on the 5,184 MedMNIST calibration samples (15% held-out, never seen during training).
3. Computes per-pixel non-conformity scores: `s(x) = |fused(x) − reference(x)|` where `reference = 0.5·(MRI + PET)`.
4. Splits pixels into **interior** vs **boundary** via Canny edge detection on the reference.
5. For each α ∈ {0.05, 0.10, 0.15}, takes the (1-α) quantile of each pool to get **regional thresholds**.
6. Evaluates **empirical coverage** on 256 unseen MedMNIST test samples — should match the target (1-α).
7. Renders a 4-row qualitative panel: MRI | PET | Fused | Residual heatmap | Uncertain pixels.

## Outputs
- `results/coverage_table.csv` — per-α target vs empirical coverage + avg interval width
- `results/sacp_scores.npz` — saved interior + boundary score arrays (so SACP can be recomputed without re-running inference, per the proposal's reproducibility requirement)
- `figures/uncertainty_heatmaps.png` — 4-sample qualitative figure

## Expected wall-clock on T4
~8–10 min (most of it is the one-time copy of `train_paired.npz` from Drive).


In [ ]:
# 0. Install deps + seeds
import os, sys, time, json, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
import csv

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE, '| Torch:', torch.__version__)


In [ ]:
# 1. Mount Drive + mirror needed files to local Colab disk (avoids FUSE drops)
from google.colab import drive
import shutil

if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/FedSafeFuse'
LOCAL_ROOT = '/content/local_fedsafe'
for sub in ['medmnist', 'partitions', 'checkpoints', 'results', 'figures']:
    os.makedirs(f'{LOCAL_ROOT}/{sub}', exist_ok=True)

# Tiny: partition + checkpoint
for fname in ['calibration_manifest.json', 'medmnist_K3.json']:
    src = f'{DRIVE_ROOT}/partitions/{fname}'
    dst = f'{LOCAL_ROOT}/partitions/{fname}'
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

CKPT_NAME = 'fedsafe_fipca_r50.pt'
ckpt_src = f'{DRIVE_ROOT}/checkpoints/{CKPT_NAME}'
ckpt_dst = f'{LOCAL_ROOT}/checkpoints/{CKPT_NAME}'
if not os.path.exists(ckpt_dst):
    shutil.copy(ckpt_src, ckpt_dst)
    print(f'Copied checkpoint: {os.path.getsize(ckpt_dst)/1e6:.0f} MB')

# Big: paired arrays
for fname in ['train_paired.npz', 'test_paired.npz']:
    src = f'{DRIVE_ROOT}/medmnist/{fname}'
    dst = f'{LOCAL_ROOT}/medmnist/{fname}'
    if not os.path.exists(dst):
        t0 = time.time()
        print(f'Copying {fname}...')
        shutil.copy(src, dst)
        print(f'  {fname}: {os.path.getsize(dst)/1e6:.0f} MB in {time.time()-t0:.1f}s')
    else:
        print(f'  {fname}: already on local disk')

PROJECT_DRIVE = LOCAL_ROOT
print('Local mirror ready')


## Embedded model + loss code (verbatim from `src/`)

In [ ]:
# FedSafe-Fuse model
"""FedSafe-Fuse: dual MobileNetV3-Small encoders + 2-layer Transformer + conv decoder.

Per the project proposal:
    - Two modality-specific encoders, each MobileNetV3-Small (~2.5M params total each)
    - Shared 2-layer Transformer module (2 heads, embed dim 128) for cross-modal attention
    - Convolutional decoder
    - Target ~8M params

Actual realised parameter count depends on decoder width; default config lands at ~5M.
"""

from __future__ import annotations

import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small


class ModalityEncoder(nn.Module):
    """MobileNetV3-Small features adapted to 1-channel input.

    MobileNetV3-Small downsamples by 32x, so a 128x128 input yields 4x4 features.
    We bilinearly upsample to 8x8 to give the cross-modal Transformer 64 tokens
    per modality (matching the proposal). The upsample is parameter-free.
    """

    def __init__(self, embed_dim: int = 128):
        super().__init__()
        base = mobilenet_v3_small(weights=None)
        # Replace first conv to accept a single channel (medical grayscale)
        base.features[0][0] = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1, bias=False)
        self.features = base.features  # (B, 576, 4, 4) for 128x128 input
        self.proj = nn.Conv2d(576, embed_dim, kernel_size=1)
        self.upsample = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.features(x)
        h = self.proj(h)
        return self.upsample(h)  # (B, embed_dim, 8, 8)


class CrossModalTransformer(nn.Module):
    """2-layer Transformer encoder over concatenated MRI/PET tokens.

    Input feature maps are flattened to 64 spatial tokens per modality, given a
    learned modality embedding, concatenated to 128 tokens, given a learned
    positional embedding, and passed through `num_layers` transformer encoder layers.
    Output is averaged across the two modality halves and reshaped back to a
    spatial feature map for the decoder.
    """

    def __init__(
        self,
        embed_dim: int = 128,
        num_layers: int = 2,
        num_heads: int = 2,
        ff_dim: int = 256,
        dropout: float = 0.1,
        spatial_tokens_per_modality: int = 64,
    ):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        total_tokens = spatial_tokens_per_modality * 2
        self.pos_embed = nn.Parameter(torch.zeros(1, total_tokens, embed_dim))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.mod_embed = nn.Embedding(2, embed_dim)  # 0 = MRI, 1 = PET
        self.spatial_tokens = spatial_tokens_per_modality

    def forward(self, mri_feat: torch.Tensor, pet_feat: torch.Tensor) -> torch.Tensor:
        # mri_feat, pet_feat: (B, C, H, W) with H*W = self.spatial_tokens
        B, C, H, W = mri_feat.shape
        mri_tokens = mri_feat.flatten(2).transpose(1, 2)  # (B, HW, C)
        pet_tokens = pet_feat.flatten(2).transpose(1, 2)
        mri_tokens = mri_tokens + self.mod_embed.weight[0]
        pet_tokens = pet_tokens + self.mod_embed.weight[1]
        tokens = torch.cat([mri_tokens, pet_tokens], dim=1) + self.pos_embed
        out = self.transformer(tokens)
        mri_out = out[:, : self.spatial_tokens].transpose(1, 2).reshape(B, C, H, W)
        pet_out = out[:, self.spatial_tokens :].transpose(1, 2).reshape(B, C, H, W)
        return 0.5 * (mri_out + pet_out)


class FusionDecoder(nn.Module):
    """Progressive bilinear upsample + conv decoder: 8x8 -> 16 -> 32 -> 64 -> 128."""

    def __init__(self, embed_dim: int = 128, base_ch: int = 256):
        super().__init__()
        c0, c1, c2, c3 = base_ch, base_ch, base_ch // 2, base_ch // 4
        self.up1 = self._block(embed_dim, c0)
        self.up2 = self._block(c0, c1)
        self.up3 = self._block(c1, c2)
        self.up4 = self._block(c2, c3)
        self.final = nn.Conv2d(c3, 1, kernel_size=1)

    @staticmethod
    def _block(in_ch: int, out_ch: int) -> nn.Sequential:
        return nn.Sequential(
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.up1(x)
        h = self.up2(h)
        h = self.up3(h)
        h = self.up4(h)
        return torch.sigmoid(self.final(h))


class FedSafeFuse(nn.Module):
    """End-to-end FedSafe-Fuse model. Inputs: two (B, 1, 128, 128) tensors. Output: (B, 1, 128, 128)."""

    def __init__(
        self,
        embed_dim: int = 128,
        transformer_layers: int = 2,
        transformer_heads: int = 2,
        transformer_ff: int = 256,
        decoder_base_ch: int = 256,
    ):
        super().__init__()
        self.mri_encoder = ModalityEncoder(embed_dim)
        self.pet_encoder = ModalityEncoder(embed_dim)
        self.fusion = CrossModalTransformer(
            embed_dim=embed_dim,
            num_layers=transformer_layers,
            num_heads=transformer_heads,
            ff_dim=transformer_ff,
        )
        self.decoder = FusionDecoder(embed_dim=embed_dim, base_ch=decoder_base_ch)

    def forward(self, mri: torch.Tensor, pet: torch.Tensor) -> torch.Tensor:
        mri_feat = self.mri_encoder(mri)
        pet_feat = self.pet_encoder(pet)
        fused_feat = self.fusion(mri_feat, pet_feat)
        return self.decoder(fused_feat)


def count_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)



In [ ]:
# Composite fusion loss + eval metrics
"""Composite fusion loss: L1 + beta * (1 - SSIM).

Implements a differentiable SSIM via a Gaussian-windowed convolution
(no external dependency on pytorch_msssim). Designed for single-channel
medical images in [0, 1].
"""

from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F


def _gaussian_window(window_size: int, sigma: float, channels: int) -> torch.Tensor:
    coords = torch.arange(window_size, dtype=torch.float32) - (window_size - 1) / 2
    g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
    g = g / g.sum()
    w_2d = g.unsqueeze(0) * g.unsqueeze(1)  # (W, W)
    return w_2d.expand(channels, 1, window_size, window_size).contiguous()


class SSIMLoss(nn.Module):
    """1 - SSIM, computed with a Gaussian sliding window. Single-channel input in [0, 1]."""

    def __init__(
        self,
        window_size: int = 11,
        sigma: float = 1.5,
        data_range: float = 1.0,
        channels: int = 1,
    ):
        super().__init__()
        self.window_size = window_size
        self.data_range = data_range
        self.C1 = (0.01 * data_range) ** 2
        self.C2 = (0.03 * data_range) ** 2
        self.register_buffer("window", _gaussian_window(window_size, sigma, channels))

    def _filter(self, x: torch.Tensor) -> torch.Tensor:
        return F.conv2d(x, self.window, padding=self.window_size // 2, groups=x.shape[1])

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        mu_p = self._filter(pred)
        mu_t = self._filter(target)
        mu_p2 = mu_p ** 2
        mu_t2 = mu_t ** 2
        mu_pt = mu_p * mu_t
        sigma_p2 = self._filter(pred * pred) - mu_p2
        sigma_t2 = self._filter(target * target) - mu_t2
        sigma_pt = self._filter(pred * target) - mu_pt
        num = (2 * mu_pt + self.C1) * (2 * sigma_pt + self.C2)
        den = (mu_p2 + mu_t2 + self.C1) * (sigma_p2 + sigma_t2 + self.C2)
        ssim_map = num / den
        return 1.0 - ssim_map.mean()


class FusionLoss(nn.Module):
    """L1(fused, ref) + beta * SSIMLoss(fused, ref).

    Reference composite is a per-pixel weighted average of source modalities, per the proposal.
    Default weights are (0.5, 0.5). Override `reference_weights` per dataset if needed.
    """

    def __init__(self, beta: float = 1.0, reference_weights: tuple[float, float] = (0.5, 0.5)):
        super().__init__()
        assert abs(sum(reference_weights) - 1.0) < 1e-6, "reference_weights must sum to 1.0"
        self.l1 = nn.L1Loss()
        self.ssim = SSIMLoss()
        self.beta = beta
        w1, w2 = reference_weights
        self.register_buffer("w1", torch.tensor(w1))
        self.register_buffer("w2", torch.tensor(w2))

    def reference(self, src1: torch.Tensor, src2: torch.Tensor) -> torch.Tensor:
        return self.w1 * src1 + self.w2 * src2

    def forward(
        self,
        fused: torch.Tensor,
        src1: torch.Tensor,
        src2: torch.Tensor,
    ) -> torch.Tensor:
        ref = self.reference(src1, src2)
        return self.l1(fused, ref) + self.beta * self.ssim(fused, ref)


@torch.no_grad()
def ssim_score(pred: torch.Tensor, target: torch.Tensor) -> float:
    """SSIM (higher is better) for evaluation. Mirrors SSIMLoss without the 1 - ... wrap."""
    loss_fn = SSIMLoss().to(pred.device)
    return float(1.0 - loss_fn(pred, target).item())


@torch.no_grad()
def psnr_score(pred: torch.Tensor, target: torch.Tensor, data_range: float = 1.0) -> float:
    """PSNR in dB."""
    mse = F.mse_loss(pred, target).item()
    if mse == 0:
        return float("inf")
    return 10.0 * torch.log10(torch.tensor(data_range ** 2 / mse)).item()


In [ ]:
# 2. Load calibration manifest + paired arrays + checkpoint
with open(f'{PROJECT_DRIVE}/partitions/calibration_manifest.json') as f:
    manifest = json.load(f)
calib_indices = manifest['medmnist']['calibration_indices']
print(f'Calibration set: {len(calib_indices):,} MedMNIST samples (15% train hold-out)')

t0 = time.time()
with np.load(f'{PROJECT_DRIVE}/medmnist/train_paired.npz') as d:
    train_mri = d['mri']
    train_pet = d['pet']
with np.load(f'{PROJECT_DRIVE}/medmnist/test_paired.npz') as d:
    test_mri = d['mri']
    test_pet = d['pet']
print(f'Arrays loaded in {time.time()-t0:.1f}s: train={train_mri.shape}, test={test_mri.shape}')

# Load the FedSafe-Fuse + FIPCA final checkpoint
model = FedSafeFuse().to(DEVICE)
ckpt = torch.load(f'{PROJECT_DRIVE}/checkpoints/{CKPT_NAME}', map_location=DEVICE)
model.load_state_dict(ckpt['state_dict'])
model.eval()
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Loaded {CKPT_NAME} (round {ckpt["round"]}, tag={ckpt["tag"]}, {n_params/1e6:.2f}M params)')


In [ ]:
# 3. SACP calibration: collect per-pixel non-conformity scores
#    s(x) = |fused(x) - reference(x)|, with reference = 0.5*(MRI + PET)
#    Pixels split into INTERIOR vs BOUNDARY via Canny edge detection on the reference.
BATCH = 16
CANNY_LOW, CANNY_HIGH = 50, 100  # per proposal
DILATE_KERNEL = np.ones((3, 3), np.uint8)  # widen edges into a 'boundary zone'
ALPHAS = [0.05, 0.10, 0.15]

print(f'Running SACP calibration on {len(calib_indices):,} samples...')
interior_chunks, boundary_chunks = [], []
t0 = time.time()
with torch.no_grad():
    for i in range(0, len(calib_indices), BATCH):
        batch_idx = calib_indices[i:i + BATCH]
        mri = torch.from_numpy(train_mri[batch_idx]).unsqueeze(1).to(DEVICE)
        pet = torch.from_numpy(train_pet[batch_idx]).unsqueeze(1).to(DEVICE)
        fused = model(mri, pet)
        ref = 0.5 * (mri + pet)
        residual = (fused - ref).abs().cpu().numpy()  # (B, 1, H, W)
        ref_np_batch = (ref.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)

        for b in range(len(batch_idx)):
            edges = cv2.Canny(ref_np_batch[b, 0], CANNY_LOW, CANNY_HIGH)
            edge_mask = cv2.dilate(edges, DILATE_KERNEL, iterations=1) > 0
            r = residual[b, 0]
            interior_chunks.append(r[~edge_mask].astype(np.float32))
            boundary_chunks.append(r[edge_mask].astype(np.float32))

interior_scores = np.concatenate(interior_chunks)
boundary_chunks = np.concatenate(boundary_chunks) if len(boundary_chunks) else np.array([], dtype=np.float32)
boundary_scores = boundary_chunks
print(f'Calibration done in {time.time()-t0:.1f}s')
print(f'  Interior pixels collected: {len(interior_scores):,}')
print(f'  Boundary pixels collected: {len(boundary_scores):,}')

# Compute per-region (1-alpha) quantile thresholds
thresholds = {}
for alpha in ALPHAS:
    thresholds[alpha] = {
        'interior': float(np.quantile(interior_scores, 1 - alpha)) if len(interior_scores) else float('nan'),
        'boundary': float(np.quantile(boundary_scores, 1 - alpha)) if len(boundary_scores) else float('nan'),
    }

print('\nRegional thresholds (smaller = tighter intervals):')
print(f'{"alpha":<8}{"interior":<14}{"boundary":<14}')
for alpha, t in thresholds.items():
    print(f'{alpha:<8}{t["interior"]:<14.4f}{t["boundary"]:<14.4f}')

# Save scores so SACP can be recomputed without retraining (reproducibility requirement)
np.savez_compressed(
    f'{PROJECT_DRIVE}/results/sacp_scores.npz',
    interior=interior_scores,
    boundary=boundary_scores,
    thresholds_alpha=np.array(ALPHAS),
    thresholds_interior=np.array([thresholds[a]['interior'] for a in ALPHAS]),
    thresholds_boundary=np.array([thresholds[a]['boundary'] for a in ALPHAS]),
)
print(f'\nSaved {PROJECT_DRIVE}/results/sacp_scores.npz')


In [ ]:
# 4. Coverage evaluation on held-out MedMNIST test set
N_TEST = 256
test_indices = list(range(N_TEST))
print(f'Evaluating coverage on first {N_TEST} test samples...')

cov_int_n = {a: 0 for a in ALPHAS}; cov_int_d = 0
cov_bnd_n = {a: 0 for a in ALPHAS}; cov_bnd_d = 0

t0 = time.time()
with torch.no_grad():
    for i in range(0, N_TEST, BATCH):
        batch_idx = test_indices[i:i + BATCH]
        mri = torch.from_numpy(test_mri[batch_idx]).unsqueeze(1).to(DEVICE)
        pet = torch.from_numpy(test_pet[batch_idx]).unsqueeze(1).to(DEVICE)
        fused = model(mri, pet)
        ref = 0.5 * (mri + pet)
        residual = (fused - ref).abs().cpu().numpy()
        ref_np_batch = (ref.cpu().numpy() * 255).clip(0, 255).astype(np.uint8)

        for b in range(len(batch_idx)):
            edges = cv2.Canny(ref_np_batch[b, 0], CANNY_LOW, CANNY_HIGH)
            edge_mask = cv2.dilate(edges, DILATE_KERNEL, iterations=1) > 0
            r = residual[b, 0]
            int_r = r[~edge_mask]
            bnd_r = r[edge_mask]
            cov_int_d += len(int_r)
            cov_bnd_d += len(bnd_r)
            for a in ALPHAS:
                cov_int_n[a] += int((int_r <= thresholds[a]['interior']).sum())
                cov_bnd_n[a] += int((bnd_r <= thresholds[a]['boundary']).sum())

print(f'Coverage eval done in {time.time()-t0:.1f}s\n')

# Build coverage table and save
rows = []
print(f'{"alpha":<8}{"target":<10}{"int_cov":<10}{"int_width":<12}{"bnd_cov":<10}{"bnd_width":<12}')
for a in ALPHAS:
    int_cov = cov_int_n[a] / max(cov_int_d, 1)
    bnd_cov = cov_bnd_n[a] / max(cov_bnd_d, 1)
    int_w = 2.0 * thresholds[a]['interior']  # symmetric interval width = 2·threshold
    bnd_w = 2.0 * thresholds[a]['boundary']
    rows.append({
        'alpha': a, 'target': 1 - a,
        'interior_coverage': int_cov, 'interior_width': int_w,
        'boundary_coverage': bnd_cov, 'boundary_width': bnd_w,
    })
    print(f'{a:<8}{1-a:<10.2f}{int_cov:<10.4f}{int_w:<12.4f}{bnd_cov:<10.4f}{bnd_w:<12.4f}')

csv_path = f'{PROJECT_DRIVE}/results/coverage_table.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    w.writeheader()
    for r in rows:
        w.writerow(r)
print(f'\nSaved {csv_path}')
print('\n(Interpretation: empirical coverage should match target = 1-alpha. '
      'Width is the size of the symmetric prediction interval per pixel.)')


In [ ]:
# 5. Qualitative uncertainty heatmaps — 4 test samples × 5 columns
SAMPLES = [0, 32, 64, 128]
ALPHA_VIZ = 0.10  # the headline alpha for the figure

t_int = thresholds[ALPHA_VIZ]['interior']
t_bnd = thresholds[ALPHA_VIZ]['boundary']

n = len(SAMPLES)
fig, axes = plt.subplots(n, 5, figsize=(16, 3.4 * n))
col_titles = ['MRI', 'PET', 'Fused (FedSafe-Fuse)', 'Residual |f-ref|', f'Uncertain pixels (α={ALPHA_VIZ})']

with torch.no_grad():
    for i, idx in enumerate(SAMPLES):
        mri = torch.from_numpy(test_mri[idx:idx+1]).unsqueeze(1).to(DEVICE)
        pet = torch.from_numpy(test_pet[idx:idx+1]).unsqueeze(1).to(DEVICE)
        fused = model(mri, pet)
        ref = 0.5 * (mri + pet)
        residual = (fused - ref).abs()

        mri_np = mri[0, 0].cpu().numpy()
        pet_np = pet[0, 0].cpu().numpy()
        fused_np = fused[0, 0].cpu().numpy()
        res_np = residual[0, 0].cpu().numpy()
        ref_u8 = (ref[0, 0].cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
        edges = cv2.Canny(ref_u8, CANNY_LOW, CANNY_HIGH)
        edge_mask = cv2.dilate(edges, DILATE_KERNEL, iterations=1) > 0

        # Per-pixel uncertain mask using regional thresholds
        unc = np.zeros_like(res_np, dtype=bool)
        unc[~edge_mask] = res_np[~edge_mask] > t_int
        unc[edge_mask]  = res_np[edge_mask] > t_bnd

        ax_row = axes[i] if n > 1 else axes
        ax_row[0].imshow(mri_np, cmap='gray', vmin=0, vmax=1)
        ax_row[1].imshow(pet_np, cmap='gray', vmin=0, vmax=1)
        ax_row[2].imshow(fused_np, cmap='gray', vmin=0, vmax=1)
        res_vmax = max(t_int, t_bnd) * 2.5
        ax_row[3].imshow(res_np, cmap='hot', vmin=0, vmax=res_vmax)
        ax_row[4].imshow(fused_np, cmap='gray', vmin=0, vmax=1)
        ax_row[4].imshow(unc.astype(np.float32), cmap='Reds', alpha=0.45, vmin=0, vmax=1)
        for j in range(5):
            ax_row[j].axis('off')
            if i == 0:
                ax_row[j].set_title(col_titles[j], fontsize=11)

plt.tight_layout()
fig_path = f'{PROJECT_DRIVE}/figures/uncertainty_heatmaps.png'
plt.savefig(fig_path, dpi=140, bbox_inches='tight')
plt.show()
print(f'Saved {fig_path}')


In [ ]:
# 6. Sync local results + figures back to Drive
import shutil
synced = 0
for sub in ['results', 'figures']:
    src_dir = f'{LOCAL_ROOT}/{sub}'
    dst_dir = f'{DRIVE_ROOT}/{sub}'
    os.makedirs(dst_dir, exist_ok=True)
    for fname in os.listdir(src_dir):
        sp = f'{src_dir}/{fname}'
        dp = f'{dst_dir}/{fname}'
        if os.path.isfile(sp):
            shutil.copy(sp, dp)
            synced += 1
print(f'Synced {synced} files to {DRIVE_ROOT}')


## When done

Paste back to chat:
1. The **coverage table** printed by cell 4 (the 3 rows showing alpha / target / interior+boundary coverage + widths).
2. The **uncertainty_heatmaps.png** figure (drag into chat).
3. Total wall-clock and any errors.

Files to download back to local repo (after sync):

| Drive path | Local destination |
|---|---|
| `results/coverage_table.csv` | `results/coverage_table.csv` |
| `results/sacp_scores.npz` | `results/sacp_scores.npz` |
| `figures/uncertainty_heatmaps.png` | `figures/uncertainty_heatmaps.png` |

The `sacp_scores.npz` lets SACP thresholds be recomputed without re-running inference, per the proposal's reproducibility requirement.
